In [3]:
import faiss
import numpy as np
from langchain.embeddings import OllamaEmbeddings

# Generate embeddings (using your existing Ollama setup)
embeddings = OllamaEmbeddings(model="mxbai-embed-large")
food_names = ["pizza", "salad", "burger"]  
vectors = np.array([embeddings.embed_query(x) for x in food_names], dtype='float32')

/opt/anaconda3/envs/blip2_env/lib/python3.9/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/var/folders/ng/fcbgt29j2v9cwpg8hztg0clr0000gn/T/ipykernel_10310/959860673.py:6: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="mxbai-embed-large")


In [ ]:
print(vectors)  # Should be (3, 4096) if using mxbai-embed-large

In [6]:
# Create FAISS index
dimension = vectors.shape[1]  # Typically 384 or 768
index = faiss.IndexFlatL2(dimension)  # Simple L2 distance index
index.add(vectors)  # Add your vectors

In [10]:
# Search (find similar foods)
query = "pepperoni pizza"
query_vector = np.array([embeddings.embed_query(query)], dtype='float32')
k = 1  # Number of neighbors
distances, indices = index.search(query_vector, k)

print(f"Most similar foods to {query}: {[food_names[i] for i in indices[0]]}")

Most similar foods to pepperoni pizza: ['pizza']


In [12]:
# Search (find similar foods)
query = "pepperoni pizza"
query_vector = np.array([embeddings.embed_query(query)], dtype='float32')
k = 1  # Number of neighbors
distances, indices = index.search(query_vector, k)

print(f"Most similar foods to {query}: {[food_names[i] for i in indices[0]]}")

# Search (find similar foods)
query = "hamburger"
query_vector = np.array([embeddings.embed_query(query)], dtype='float32')
k = 1  # Number of neighbors
distances, indices = index.search(query_vector, k)

print(f"Most similar foods to {query}: {[food_names[i] for i in indices[0]]}")

Most similar foods to pepperoni pizza: ['pizza']
Most similar foods to hamburger: ['burger']


/opt/anaconda3/envs/blip2_env/lib/python3.9/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


image 1/1 /Users/sreehari/Desktop/university/My Uni/Project/project-learning/week3/test_images/plate1.jpeg: 640x640 1 bowl, 1 broccoli, 1 dining table, 53.7ms
Speed: 1.5ms preprocess, 53.7ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)


# Nutrition DB to FAISS


In [8]:
import sqlite3
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

# Load an embedding model (small but effective)
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Connect to SQLite DB
conn = sqlite3.connect("./nutrition.db")
cursor = conn.cursor()
cursor.execute("SELECT name, calories, protein, carbs FROM foods")
foods = cursor.fetchall()

# Extract names and nutrition data
food_names = [food[0] for food in foods]
nutrition_data = [food[1:] for food in foods]  # (calories, protein, carbs)

# Generate embeddings for food names
name_embeddings = embedding_model.encode(food_names, normalize_embeddings=True)

# Build FAISS index
dimension = name_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # Inner product for cosine similarity
index.add(name_embeddings)

# Save index and metadata for later use
faiss.write_index(index, "nutrition_faiss.index")
np.save("food_names.npy", food_names)
np.save("nutrition_data.npy", nutrition_data)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]